In [8]:
from RKDRetriever import *
import pandas as pd

In [ ]:
data_path = "data/"
companies_path = "CAC40.csv"

start_date = "2025-09-27"
end_date   = "2025-10-29"

RKD = RKDRetriever()
RKD.CreateAuthorization()

companies = pd.read_csv(companies_path)
RIC = companies["RIC"].tolist()

In [ ]:
def drop_timezone(data):
    data["TIMESTAMP"] = pd.to_datetime(data["TIMESTAMP"], utc=True).dt.tz_localize(None)
    data.rename(columns={"TIMESTAMP": "DATE"}, inplace=True)
    data.set_index("DATE", inplace=True)
    return data

In [ ]:
# Generate date list as strings YYYY-MM-DD
dates = pd.date_range(start=start_date, end=end_date, freq="D").strftime("%Y-%m-%d").tolist()

# Prepare result DataFrame with RIC as index and date columns
result = pd.DataFrame(index=RIC, columns=dates)

In [ ]:

tmp = 1
for ric in RIC:
    
    try:
        print(tmp, f"Retrieving data for {ric}...")
        tmp += 1
        data = RKD.RetrieveInterday(ric, start_date + "T00:00:00", end_date + "T23:59:59",fields='CLOSE')
        data = drop_timezone(data)
        
        data["date_only"] = data.index.date
        daily_close = data.groupby("date_only")["CLOSE"].last()
        for dt in dates:
            d = pd.to_datetime(dt).date()
            result.at[ric, dt] = daily_close.get(d, pd.NA)
    except Exception as e:
        print(f"Error retrieving data for {ric}: {e}")

1 Retrieving data for LVMH.PA...
Interday request success
2 Retrieving data for SCHN.PA...
Interday request success
3 Retrieving data for TTEF.PA...
Interday request success
4 Retrieving data for SASY.PA...
Interday request success
5 Retrieving data for AIR.PA...
Interday request success
6 Retrieving data for AIRP.PA...
Interday request success
7 Retrieving data for OREP.PA...
Interday request success
8 Retrieving data for SAF.PA...
Interday request success
9 Retrieving data for ESLX.PA...
Interday request success
10 Retrieving data for HRMS.PA...
Interday request success
11 Retrieving data for AXAF.PA...
Interday request success
12 Retrieving data for BNPP.PA...
Interday request success
13 Retrieving data for SGEF.PA...
Interday request success
14 Retrieving data for DANO.PA...
Interday request success
15 Retrieving data for SGOB.PA...
Interday request success
16 Retrieving data for ENGIE.PA...
Interday request success
17 Retrieving data for STLAM.MI...
Interday request success
18 Ret

In [ ]:
result.dropna(axis=1, how='all', inplace=True)
# Reset index and save to CSV
output_file = os.path.join(data_path, f"prices_{start_date}_{end_date}.csv")
result.reset_index(inplace=True)
result.rename(columns={"index": "RIC"}, inplace=True)
result.to_csv(output_file, index=False)

print(f"Saved consolidated prices to {output_file}")

Saved consolidated prices to data/prices_2025-09-04_2025-10-29.csv
